# 🐍 Day 5: Flask Auth

- Flask-Login for session management
- Password hashing (werkzeug/bcrypt)
- Login and logout flows
- Protecting routes

## Password Hashing

Never store plain passwords. Use `werkzeug.security.generate_password_hash` and `check_password_hash`.

In [ ]:
from werkzeug.security import generate_password_hash, check_password_hash

password = "secret123"
hashed = generate_password_hash(password, method="pbkdf2:sha256")
print("Hashed:", hashed[:50] + "...")

print("Valid:", check_password_hash(hashed, "secret123"))
print("Invalid:", check_password_hash(hashed, "wrong"))

## User Model for Flask-Login

Implement `UserMixin` or provide: `is_authenticated`, `is_active`, `is_anonymous`, `get_id()`.

In [ ]:
from flask_login import UserMixin

class User(UserMixin):
    def __init__(self, id, username, password_hash):
        self.id = id
        self.username = username
        self.password_hash = password_hash

    def check_password(self, password):
        return check_password_hash(self.password_hash, password)

user = User(1, "alice", generate_password_hash("secret"))
print("UserMixin provides: is_authenticated, get_id, etc.")

## Flask-Login Setup

`pip install flask-login`. Initialize LoginManager, set user_loader.

In [ ]:
from flask import Flask
from flask_login import LoginManager

app = Flask(__name__)
app.config["SECRET_KEY"] = "dev-secret"

login_manager = LoginManager()
login_manager.init_app(app)
login_manager.login_view = "auth.login"  # Redirect if not logged in

@login_manager.user_loader
def load_user(user_id):
    # Load user from DB by id; return None if not found
    return None  # Placeholder

print("LoginManager + user_loader required")

## Login Route

Use `login_user(user)` and `logout_user()` from flask_login.

In [ ]:
from flask import request, redirect, url_for
from flask_login import login_user, logout_user, login_required, current_user

@app.route("/login", methods=["GET", "POST"])
def login():
    if request.method == "POST":
        username = request.form.get("username")
        password = request.form.get("password")
        user = get_user_by_username(username)  # Your DB lookup
        if user and user.check_password(password):
            login_user(user)
            return redirect(url_for("dashboard"))
    return "Login form"

@app.route("/logout")
@login_required
def logout():
    logout_user()
    return redirect(url_for("login"))

def get_user_by_username(name):
    return None  # Placeholder

print("login_user(), logout_user(), @login_required")

## Protecting Routes

Use `@login_required` decorator. Access current user with `current_user`.

In [ ]:
@app.route("/dashboard")
@login_required
def dashboard():
    return f"Hello, {current_user.username}!"

# In template: {% if current_user.is_authenticated %}...{% endif %}
print("current_user is available in routes and templates")

## Optional: Bcrypt

For stronger hashing: `pip install bcrypt`. Flask-Bcrypt or use bcrypt directly.

In [ ]:
# Using bcrypt (if installed)
try:
    import bcrypt
    pwd = b"secret"
    hashed = bcrypt.hashpw(pwd, bcrypt.gensalt())
    print("Bcrypt:", bcrypt.checkpw(pwd, hashed))
except ImportError:
    print("bcrypt not installed; werkzeug is fine for learning")

## Session and Cookies

Flask-Login stores user id in encrypted session cookie. Session is server-side (or client cookie with secret).

In [ ]:
from flask import session

# Session is a dict; Flask-Login uses it internally
# SECRET_KEY is used to sign session cookie
print("SECRET_KEY must be set for secure sessions")

## Common Mistakes

- **Storing plain passwords**: Always hash with generate_password_hash
- **Weak SECRET_KEY**: Use strong random key in production
- **Not implementing user_loader**: Flask-Login needs it to restore user from session
- **Forgetting login_view**: Users get 401 without redirect to login page
- **Using GET for login**: Prefer POST for credentials

## Exercises

In [ ]:
# Exercise 1: Hash 'mypassword' and verify with check_password_hash.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Create a User class with UserMixin, id, username, password_hash. Add check_password method.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Implement user_loader that returns User(id=1) for id '1', else None.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Add @login_required to a route. What happens when unauthenticated user visits?
# YOUR CODE HERE

In [ ]:
# Exercise 5: In a template, show 'Logout' if current_user.is_authenticated else 'Login'.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Add remember=True to login_user(). What does it do?
# YOUR CODE HERE

## Key Takeaways

- generate_password_hash / check_password_hash for passwords
- Flask-Login: UserMixin, user_loader, login_user, logout_user, @login_required
- current_user in routes and templates
- SECRET_KEY required for sessions

## 🔜 Next: Day 6 - Flask REST API

Tomorrow: RESTful endpoints, jsonify, request parsing, and error handlers!